# Member 2: Multi-Person Tracking Model Training (Google Colab)
### Project: AI-Based Real-Time Shoplifting Detection System (`J26-IT-307`)
**Component 2 Responsibility**: Multi-Person Tracking & Retail Zone Analytics.

---
### Overview of Member 2 Pipeline:
1. **Detector Model**: YOLOv8 fine-tuned on single-class `person` in retail CCTV surveillance scenes.
2. **Tracker Algorithm**: ByteTrack (Two-stage association: high-confidence detections + low-confidence occlusion recovery with Kalman filter & CIoU matching).
3. **Output Interface**: API-contract JSON sequence (`frame_id`, `tracks[].person_id`, `bbox`, `center`) fed to Member 3 (Behaviour Analysis).

> **Why Colab GPU?** Training on local CPU takes ~11+ minutes per epoch (~18 hours for 100 epochs). On Google Colab NVIDIA T4 GPU, 50 epochs takes only ~15 minutes!

## Step 1: Check GPU Runtime & PyTorch Environment
Ensure Colab runtime is set to GPU: **Runtime -> Change runtime type -> T4 GPU**.

In [ ]:
# Check GPU status
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## Step 2: Install Ultralytics & Tracking Dependencies

In [ ]:
!pip install -q ultralytics lapx filterpy opencv-python-headless
import ultralytics
ultralytics.checks()

## Step 3: Load Dataset (Google Drive or Direct Upload)
You can either:
- **Option A (Recommended)**: Upload `member2_dataset_colab.zip` to Google Drive and mount it (avoids uploading every session).
- **Option B**: Drag and drop `member2_dataset_colab.zip` into the Colab Files tab on the left.

In [ ]:
import os
from pathlib import Path

# If using Google Drive, uncomment the two lines below:
# from google.colab import drive
# drive.mount('/content/drive')
# ZIP_SRC = '/content/drive/MyDrive/member2_dataset_colab.zip'

ZIP_SRC = '/content/member2_dataset_colab.zip'
EXTRACT_DIR = '/content/data'

if os.path.exists(ZIP_SRC):
    print(f"Unzipping {ZIP_SRC} into {EXTRACT_DIR}...")
    !unzip -q {ZIP_SRC} -d {EXTRACT_DIR}
    print("Extraction complete!")
else:
    print(f"⚠️ Please upload member2_dataset_colab.zip to /content or update ZIP_SRC")

# Verify images
train_imgs = list(Path('/content/data/dataset/images/train').glob('*.*'))
val_imgs = list(Path('/content/data/dataset/images/val').glob('*.*'))
print(f"Dataset verified: {len(train_imgs)} train images, {len(val_imgs)} val images.")

## Step 4: Configure `data.yaml`
Create the YOLO dataset config file pointing to `/content/data/dataset`.

In [ ]:
import yaml

data_config = {
    'path': '/content/data/dataset',
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'person'
    }
}

yaml_path = '/content/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print('Generated /content/data.yaml:')
!cat /content/data.yaml

## Step 5: Fine-Tune the YOLOv8 Person Detector
We fine-tune `yolov8n.pt` with hyperparameters tailored for retail surveillance:
- **`single_cls=True`**: All capacity is focused on the `person` class
- **`mosaic=1.0`**: Occlusion simulation for crowded shopping aisles
- **`mixup=0.1`**: Simulates overlapping people
- **`degrees=10.0`**: Angled/overhead CCTV perspective adjustment
- **`patience=15`**: Early stopping to prevent overfitting

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/data.yaml',
    epochs=50,          # 50 epochs recommended (fast and high accuracy on T4)
    batch=16,
    imgsz=640,
    device=0,
    project='/content/runs/detect',
    name='person_bytetrack',
    exist_ok=True,
    patience=15,
    workers=4,
    pretrained=True,
    single_cls=True,
    mosaic=1.0,
    mixup=0.1,
    degrees=10.0,
    plots=True
)

## Step 6: Evaluate Model & Display Metrics
Inspect the training convergence curves and evaluate on the validation set.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
from pathlib import Path

run_dir = Path('/content/runs/detect/person_bytetrack')

for chart in ['results.png', 'confusion_matrix.png', 'BoxF1_curve.png', 'BoxPR_curve.png']:
    chart_path = run_dir / chart
    if chart_path.exists():
        print(f'=== {chart} ===')
        display(Image(str(chart_path), width=750))

# Quantitative Validation
best_weights = run_dir / 'weights' / 'best.pt'
val_model = YOLO(str(best_weights))
val_metrics = val_model.val(data='/content/data.yaml', split='val')
print('\n' + '='*40)
print('Final Validation Performance:')
print(f'mAP@50:     {val_metrics.box.map50:.4f}')
print(f'mAP@50-95:  {val_metrics.box.map:.4f}')
print(f'Precision:  {val_metrics.box.mp:.4f}')
print(f'Recall:     {val_metrics.box.mr:.4f}')
print('='*40)

## Step 7: Test Multi-Person Tracking (ByteTrack) on CCTV Footage
Test the trained person detector with ByteTrack association on a test video clip to verify persistent person tracking.

In [ ]:
from pathlib import Path

video_candidates = list(Path('/content/data/videos').glob('*.mp4'))
if not video_candidates:
    print('⚠️ No test videos found under /content/data/videos. Skipping video test.')
else:
    test_video = str(video_candidates[0])
    print(f'Running ByteTrack tracking on: {test_video}')
    
    track_results = val_model.track(
        source=test_video,
        tracker='bytetrack.yaml',
        conf=0.15,       # Low threshold for occlusion recovery
        iou=0.5,
        classes=[0],     # Person class only
        save=True,
        project='/content/runs/track',
        name='demo',
        exist_ok=True
    )
    out_path = f'/content/runs/track/demo/{Path(test_video).name}'
    print(f'Tracking complete! Annotated video: {out_path}')

## Step 8: Package & Download Model for Member 2 Local Project
This step bundles `best.pt` (renamed as `yolov8n-person-track.pt`), `results.csv`, and validation charts into a downloadable zip file.

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

export_dir = Path('/content/member2_trained_model')
export_dir.mkdir(parents=True, exist_ok=True)

run_dir = Path('/content/runs/detect/person_bytetrack')
best_pt = run_dir / 'weights' / 'best.pt'

if best_pt.exists():
    shutil.copy2(best_pt, export_dir / 'yolov8n-person-track.pt')
    print('Exported: yolov8n-person-track.pt')

for f in ['results.csv', 'results.png', 'confusion_matrix.png', 'BoxF1_curve.png', 'args.yaml']:
    src = run_dir / f
    if src.exists():
        shutil.copy2(src, export_dir / f)

zip_out = '/content/member2_trained_model'
shutil.make_archive(zip_out, 'zip', export_dir)
print(f'Archive created: {zip_out}.zip')

# Trigger direct download in browser
files.download(f'{zip_out}.zip')
print('Download initiated! Once downloaded, extract files into member2-person-tracking/models/ and outputs/')
